# Heap & Monotonic Stack — Cheat Sheet

**Companion to:** Trees & Graphs Study Guide, DP Study Guide

**What this adds:** both topics were referenced in the study guides but not fully covered. This cheat sheet gives you the templates, Python API quirks, and problem patterns you need under interview conditions.

---
## When to Use

| Signal | Pattern to reach for |
| :--- | :--- |
| "kth largest", "kth smallest", "top K elements" | Heap |
| "median from data stream", "running median" | Two Heaps |
| "merge K sorted lists/arrays" | Heap (K-way merge) |
| "next greater element", "daily temperatures" | Monotonic Stack (decreasing) |
| "next smaller element", "previous smaller" | Monotonic Stack (increasing) |
| "largest rectangle", "trap rainwater" | Monotonic Stack |
| "sliding window maximum" | Monotonic Deque |

---
## Patterns at a Glance

| Pattern | Key idea | Time | Problems |
| :--- | :--- | :--- | :--- |
| Min Heap | Always access the smallest element in O(1) | Push/pop O(log n) | 215, 347, 373 |
| Max Heap | Negate values — Python only has min-heap | Push/pop O(log n) | 215, 1046 |
| Two Heaps | Split stream into lower/upper half | Push/pop O(log n) | 295, 480 |
| K-way Merge | Heap of (value, list_index, element_index) | O(n log k) | 23, 378 |
| Monotonic Stack (decreasing) | Pop when current > top → next greater element | O(n) | 496, 739, 84 |
| Monotonic Stack (increasing) | Pop when current < top → next smaller element | O(n) | 85, 42 |
| Monotonic Deque | Evict from front when outside window; pop from back when not useful | O(n) | 239 |

---
# Part 0 — Quick Recap

**Heap** is a complete binary tree maintained so the root is always the minimum (min-heap) or maximum (max-heap). It gives O(1) access to the extreme element and O(log n) insertion/deletion. Use it whenever you need repeated access to the smallest or largest element in a changing collection.

**Monotonic Stack** is a stack that maintains a strictly increasing or decreasing order of elements. Whenever you push a new element, you pop everything that violates the order first. Use it whenever the problem asks about the next/previous greater/smaller element, or spans/rectangles involving heights.

**How they relate:** both deal with ordering and extremes in a sequence, but in different directions — Heap gives you the global extreme repeatedly; Monotonic Stack gives you the local next/previous extreme for each element.

**Closest neighbors to distinguish from:**
- Heap vs Sorting: use Heap when you need the top-K elements from a stream or don't need full sort order
- Monotonic Stack vs Two Pointers: use Monotonic Stack when the answer for each element depends on the nearest element that is greater/smaller, not just a pair

---
# Part 1 — Heap

## Core idea

Python's `heapq` is a **min-heap only**. To simulate a max-heap, negate all values on push and negate again on pop. For heaps of tuples, Python compares element by element — put the priority value first in the tuple.

In [ ]:
import heapq

# ── Min-heap API ─────────────────────────────────────────────────────────────
heap = []
heapq.heappush(heap, val)           # push
val = heapq.heappop(heap)           # pop smallest
val = heap[0]                       # peek smallest — O(1), no pop
heap = heapq.nsmallest(k, iterable) # k smallest — use for one-off queries
heapq.heapify(list)                 # convert list to heap in-place — O(n)

# ── Max-heap — negate values ──────────────────────────────────────────────────
heapq.heappush(heap, -val)          # push negated
val = -heapq.heappop(heap)          # pop and negate back

# ── Heap of tuples — priority goes FIRST ─────────────────────────────────────
heapq.heappush(heap, (priority, item))  # smallest priority popped first

# ── Template: Kth Largest Element (LC 215) ───────────────────────────────────
# Maintain a min-heap of size k — root is always the kth largest
def findKthLargest(nums, k):
    heap = []
    for num in nums:
        heapq.heappush(heap, num)
        if len(heap) > k:
            heapq.heappop(heap)     # evict smallest — keeps k largest
    return heap[0]                  # root = kth largest

# ── Template: Top K Frequent Elements (LC 347) ───────────────────────────────
from collections import Counter
def topKFrequent(nums, k):
    count = Counter(nums)
    return heapq.nlargest(k, count.keys(), key=count.get)

## Two Heaps pattern

Split a stream into a **lower half** (max-heap) and **upper half** (min-heap). The median is always derivable from the two roots. Rebalance after every insertion so sizes differ by at most 1.

In [ ]:
# Template: Find Median from Data Stream (LC 295)
class MedianFinder:
    def __init__(self):
        self.lo = []    # max-heap (negated) — lower half
        self.hi = []    # min-heap — upper half

    def addNum(self, num):
        heapq.heappush(self.lo, -num)               # always push to lower half first
        heapq.heappush(self.hi, -heapq.heappop(self.lo))  # balance: move max of lo to hi
        if len(self.lo) < len(self.hi):             # keep lo >= hi in size
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def findMedian(self):
        if len(self.lo) > len(self.hi):
            return -self.lo[0]
        return (-self.lo[0] + self.hi[0]) / 2

## K-way Merge pattern

In [ ]:
# Template: Merge K Sorted Lists (LC 23)
# Push (value, list_index, element_index) — heap compares tuples left to right
def mergeKLists(lists):
    heap = []
    for i, node in enumerate(lists):
        if node:
            heapq.heappush(heap, (node.val, i, node))

    dummy = cur = ListNode(0)
    while heap:
        val, i, node = heapq.heappop(heap)
        cur.next = node
        cur = cur.next
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))

## Common mistakes

- Forgetting to negate when simulating a max-heap — pushing `val` instead of `-val`
- Peeking with `heap[0]` on an empty heap — always check `if heap` first
- Using `heapq.nlargest(k, ...)` when you need repeated queries — it's O(n log k) each call; maintain a heap instead

**Practice problems:**

| Problem | Pattern | Notes |
| :--- | :--- | :--- |
| LC 215 — Kth Largest Element | Min-heap of size k | Root = kth largest |
| LC 347 — Top K Frequent Elements | Max-heap on frequency | Use `Counter` + `nlargest` |
| LC 295 — Find Median from Data Stream | Two Heaps | Always rebalance after push |
| LC 23 — Merge K Sorted Lists | K-way merge | Tuple heap: `(val, list_idx, node)` |

---
# Part 2 — Monotonic Stack

## Core idea

Maintain a stack in strictly increasing or decreasing order. When a new element violates the order, pop elements until it doesn't — the popped elements have found their "next greater/smaller" answer. The direction of the stack determines what question you're answering.

In [ ]:
# ── Monotonic Stack (decreasing) → Next Greater Element ──────────────────────
# Stack holds indices of elements waiting for their next greater element
# Pop when current element is greater than stack top
def nextGreaterElement(nums):
    n = len(nums)
    res = [-1] * n
    stack = []                          # stores indices, monotonically decreasing values
    for i in range(n):
        while stack and nums[i] > nums[stack[-1]]:  # current breaks decreasing order
            idx = stack.pop()
            res[idx] = nums[i]          # nums[i] is the next greater for idx
        stack.append(i)
    return res                          # remaining indices in stack have no next greater


# ── Monotonic Stack (increasing) → Next Smaller Element ──────────────────────
def nextSmallerElement(nums):
    n = len(nums)
    res = [-1] * n
    stack = []
    for i in range(n):
        while stack and nums[i] < nums[stack[-1]]:  # current breaks increasing order
            idx = stack.pop()
            res[idx] = nums[i]
        stack.append(i)
    return res


# ── Template: Daily Temperatures (LC 739) ────────────────────────────────────
# How many days until a warmer temperature?
def dailyTemperatures(temps):
    res = [0] * len(temps)
    stack = []                          # indices of unresolved days
    for i, t in enumerate(temps):
        while stack and t > temps[stack[-1]]:
            idx = stack.pop()
            res[idx] = i - idx          # distance to next warmer day
        stack.append(i)
    return res


# ── Template: Largest Rectangle in Histogram (LC 84) ─────────────────────────
# Increasing stack — pop when current bar is shorter than top
def largestRectangleArea(heights):
    heights.append(0)                   # sentinel to flush remaining stack at end
    stack = [-1]                        # sentinel index
    res = 0
    for i, h in enumerate(heights):
        while stack[-1] != -1 and h < heights[stack[-1]]:
            height = heights[stack.pop()]
            width  = i - stack[-1] - 1  # width between current and new top
            res = max(res, height * width)
        stack.append(i)
    return res

## Monotonic Deque — Sliding Window Maximum

In [ ]:
from collections import deque

# Template: Sliding Window Maximum (LC 239)
# Deque stores indices in decreasing value order
# Front = index of max in current window
def maxSlidingWindow(nums, k):
    dq  = deque()                       # indices, values decreasing front to back
    res = []
    for i, num in enumerate(nums):
        while dq and nums[dq[-1]] < num:    # remove smaller elements from back
            dq.pop()
        dq.append(i)
        if dq[0] < i - k + 1:              # evict front if outside window
            dq.popleft()
        if i >= k - 1:                      # window fully formed
            res.append(nums[dq[0]])
    return res

## Variants

- **Circular array:** iterate `2n` times using `i % n` to handle wrap-around (LC 503)
- **Previous greater element:** iterate right to left instead of left to right
- **Stack stores values instead of indices:** use when you only need the answer value, not the distance

## Common mistakes

- Storing values instead of indices when you need to compute distances or widths
- Forgetting the sentinel (`-1` or appending `0`) in histogram problems — stack never fully empties without it
- Using a deque but forgetting to evict the front when it falls outside the window

**Practice problems:**

| Problem | Stack type | Notes |
| :--- | :--- | :--- |
| LC 739 — Daily Temperatures | Decreasing | Store indices; answer = `i - idx` |
| LC 496 — Next Greater Element I | Decreasing | Use hashmap to map value to next greater |
| LC 84 — Largest Rectangle in Histogram | Increasing | Append sentinel 0; use `-1` as stack base |
| LC 42 — Trapping Rain Water | Increasing | Can also solve with two pointers |
| LC 239 — Sliding Window Maximum | Decreasing deque | Evict front when outside window |

---
# Decision Guide

```
Do you need repeated access to the min or max of a changing collection?
├── Yes → Heap
│         ├── Top K elements from a stream       → Min-heap of size k
│         ├── Median from a stream               → Two Heaps (lo max-heap, hi min-heap)
│         └── Merge K sorted lists               → K-way merge (tuple heap)
│
└── No  → Does each element need to know its nearest greater/smaller neighbor?
          ├── Yes → Monotonic Stack
          │         ├── Next GREATER element      → Decreasing stack (pop when cur > top)
          │         ├── Next SMALLER element      → Increasing stack (pop when cur < top)
          │         ├── Spans / rectangles        → Increasing stack + sentinel
          │         └── Sliding window maximum    → Monotonic Deque
          │                                          (evict front if outside window)
          └── No  → Neither — reconsider Two Pointers or Sliding Window
```